# Murphy 1 · Chapter 10 · Practice

From **Neuron Voxel Lab** (https://neuron-voxel-lab.vercel.app/#M10). Each exercise has a question, a hint, and an answer that stays hidden until you click.

**How to use:** try the starter cell first (fill in the `...` TODOs), then compare with the hidden solution cell (double-click its title to see the code), then open the answer.

## 10.1 · Logistic regression from scratch  
*Section §10.2 · code*

Make two overlapping 2-D Gaussian blobs (200 points each). Fit logistic regression with gradient descent on the mean negative log-likelihood (lr 0.5, 500 steps). Print the weights, the final loss, and the training accuracy, and compare with scikit-learn's LogisticRegression (no penalty).

<details><summary><b>Hint</b></summary>

Gradient = Xᵀ(σ(Xw) − y)/N with a column of ones in X for the bias.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)
X = np.vstack([rng.normal([-1, -1], 1, (200, 2)), rng.normal([1, 1.2], 1, (200, 2))]); y = np.r_[np.zeros(200), np.ones(200)]
A = np.column_stack([np.ones(400), X]); w = np.zeros(3)
sig = lambda z: 1 / (1 + np.exp(-z))
for _ in range(500):
    p = sig(A @ w)
    w -= ...  # TODO: gradient of the mean NLL
p = sig(A @ w); nll = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
sk = LogisticRegression(penalty=None).fit(X, y)
print("ours   w =", w.round(3), f" loss {nll:.4f}  acc {np.mean((p > .5) == y):.3f}")
print("sklearn w =", np.r_[sk.intercept_, sk.coef_[0]].round(3), f" acc {sk.score(X, y):.3f}")

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
from sklearn.linear_model import LogisticRegression
rng = np.random.default_rng(0)
X = np.vstack([rng.normal([-1, -1], 1, (200, 2)), rng.normal([1, 1.2], 1, (200, 2))]); y = np.r_[np.zeros(200), np.ones(200)]
A = np.column_stack([np.ones(400), X]); w = np.zeros(3)
sig = lambda z: 1 / (1 + np.exp(-z))
for _ in range(500):
    p = sig(A @ w)
    w -= 0.5 * A.T @ (p - y) / len(y)  # gradient of the mean NLL
p = sig(A @ w); nll = -np.mean(y * np.log(p) + (1 - y) * np.log(1 - p))
sk = LogisticRegression(penalty=None).fit(X, y)
print("ours   w =", w.round(3), f" loss {nll:.4f}  acc {np.mean((p > .5) == y):.3f}")
print("sklearn w =", np.r_[sk.intercept_, sk.coef_[0]].round(3), f" acc {sk.score(X, y):.3f}")

<details><summary><b>Answer and explanation</b></summary>

Your weights match scikit-learn's to about 2 decimals, and so does the accuracy. Nothing magic happens inside the library: it minimises the same convex loss, just with a faster optimiser (Murphy §10.2.3).

Expected output:
```
ours   w = [-0.107  1.853  1.976]  loss 0.1817  acc 0.912
sklearn w = [-0.109  1.854  1.976]  acc 0.910
```

</details>

## 10.2 · Logistic regression is convex  
*Section §10.2 · derive*

For NLL(w) = −Σ [yₙ log μₙ + (1 − yₙ) log(1 − μₙ)] with μₙ = σ(wᵀxₙ), show the gradient is Σ(μₙ − yₙ)xₙ and the Hessian is Σ μₙ(1 − μₙ) xₙxₙᵀ. Why does that make the loss convex?

<details><summary><b>Hint</b></summary>

Use dμ/dz = μ(1 − μ) (exercise 2.5) and the chain rule.

</details>

✍️ *Write your answer here (double-click to edit).*

<details><summary><b>Worked answer</b></summary>

d/dz of −[y log μ + (1−y) log(1−μ)] = −[y(1−μ) − (1−y)μ] = μ − y; times dz/dw = x gives (μ − y)x. Differentiating again: d(μ − y)/dw = μ(1−μ)x, so H = Σ μ(1−μ) x xᵀ. For any vector v, vᵀHv = Σ μ(1−μ)(vᵀx)² ≥ 0 because μ(1−μ) > 0. A Hessian that's never negative means the bowl curves upward in every direction: one global minimum and no traps (Murphy §10.2.3.4).

</details>

## 10.3 · The perceptron: learning from mistakes  
*Section §10.2 · code*

On linearly separable 2-D data, run the perceptron algorithm: for each misclassified point, add yₙxₙ to w (labels ±1). Count the mistakes per pass until a pass has none.

<details><summary><b>Hint</b></summary>

Predict sign(wᵀx). Only update on mistakes.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(1)
X = np.vstack([rng.normal([-2, -2], 0.8, (50, 2)), rng.normal([2, 2], 0.8, (50, 2))])
y = np.r_[-np.ones(50), np.ones(50)]; A = np.column_stack([np.ones(100), X]); w = np.zeros(3)
for epoch in range(20):
    mistakes = 0
    for i in rng.permutation(100):
        if y[i] * (A[i] @ w) <= 0:
            w += ...  # TODO: nudge the line toward the misclassified point
            mistakes += 1
    print(f"pass {epoch + 1}: {mistakes} mistakes")
    if mistakes == 0: break
print("w =", w.round(2))

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(1)
X = np.vstack([rng.normal([-2, -2], 0.8, (50, 2)), rng.normal([2, 2], 0.8, (50, 2))])
y = np.r_[-np.ones(50), np.ones(50)]; A = np.column_stack([np.ones(100), X]); w = np.zeros(3)
for epoch in range(20):
    mistakes = 0
    for i in rng.permutation(100):
        if y[i] * (A[i] @ w) <= 0:
            w += y[i] * A[i]  # nudge the line toward the misclassified point
            mistakes += 1
    print(f"pass {epoch + 1}: {mistakes} mistakes")
    if mistakes == 0: break
print("w =", w.round(2))

<details><summary><b>Answer and explanation</b></summary>

Mistakes drop to 0 after a few passes: on separable data the perceptron is guaranteed to stop. It's logistic regression's SGD with the sigmoid replaced by a hard step, which is why it has no notion of confidence and never settles on a 'best' line, only a valid one (Murphy §10.2.5).

Expected output:
```
pass 1: 1 mistakes
pass 2: 0 mistakes
w = [1.   1.98 2.14]
```

</details>

## 10.4 · Newton's method (IRLS) vs gradient descent  
*Section §10.2 · code*

Fit logistic regression on the 10.1 data with Newton's method: w ← w − H⁻¹g using the Hessian from 10.2. Print the loss after each of 6 iterations, and compare with how many GD steps reach the same loss.

<details><summary><b>Hint</b></summary>

H = Aᵀ diag(μ(1−μ)) A / N, g = Aᵀ(μ − y)/N.

</details>

In [ ]:
# Your turn: replace each ... with code
import numpy as np
rng = np.random.default_rng(0)
X = np.vstack([rng.normal([-1, -1], 1, (200, 2)), rng.normal([1, 1.2], 1, (200, 2))]); y = np.r_[np.zeros(200), np.ones(200)]
A = np.column_stack([np.ones(400), X]); sig = lambda z: 1 / (1 + np.exp(-z))
nll = lambda w: -np.mean(y * np.log(sig(A @ w)) + (1 - y) * np.log(1 - sig(A @ w)))
w = np.zeros(3)
for it in range(6):
    mu = sig(A @ w); g = A.T @ (mu - y) / 400
    H = ...  # TODO: curvature
    w -= ...  # TODO: Newton step
    print(f"Newton iter {it + 1}: loss {nll(w):.6f}")
target = nll(w) + 1e-4; wg = np.zeros(3); steps = 0
while nll(wg) > target:
    wg -= 0.5 * A.T @ (sig(A @ wg) - y) / 400; steps += 1
print("gradient descent steps to get within 1e-4:", steps)

In [ ]:
# @title Solution (double-click to show the code)
import numpy as np
rng = np.random.default_rng(0)
X = np.vstack([rng.normal([-1, -1], 1, (200, 2)), rng.normal([1, 1.2], 1, (200, 2))]); y = np.r_[np.zeros(200), np.ones(200)]
A = np.column_stack([np.ones(400), X]); sig = lambda z: 1 / (1 + np.exp(-z))
nll = lambda w: -np.mean(y * np.log(sig(A @ w)) + (1 - y) * np.log(1 - sig(A @ w)))
w = np.zeros(3)
for it in range(6):
    mu = sig(A @ w); g = A.T @ (mu - y) / 400
    H = (A * (mu * (1 - mu))[:, None]).T @ A / 400  # curvature
    w -= np.linalg.solve(H, g)  # Newton step
    print(f"Newton iter {it + 1}: loss {nll(w):.6f}")
target = nll(w) + 1e-4; wg = np.zeros(3); steps = 0
while nll(wg) > target:
    wg -= 0.5 * A.T @ (sig(A @ wg) - y) / 400; steps += 1
print("gradient descent steps to get within 1e-4:", steps)

<details><summary><b>Answer and explanation</b></summary>

Newton is within 10⁻⁶ of the minimum after 5 iterations; gradient descent needs about 160 steps just to get within 10⁻⁴. Newton uses the curvature to jump to the bottom of the local quadratic bowl. Each Newton step costs more (it solves a D×D system), which is why it's used for small models like this, not for neural nets with millions of weights. Murphy calls it IRLS (§10.2.6).

Expected output:
```
Newton iter 1: loss 0.293433
Newton iter 2: loss 0.213236
Newton iter 3: loss 0.187476
Newton iter 4: loss 0.182061
Newton iter 5: loss 0.181673
Newton iter 6: loss 0.181670
gradient descent steps to get within 1e-4: 162
```

</details>